## Now we need our Language Encoder to contrast to our SiglipVisionModel

In [5]:
IMAGENET_STANDARD_MEAN = [0.5, 0.5, 0.5]
IMAGENET_STANDARD_STD = [0.5, 0.5, 0.5]

* adding tokens to the image:
```python
tokens = [image_tokens...,
       BOS, prefix_tokens..., SEP
        suffix_tokens..., EOS, PAD...]
```

* image_tokens -->
* The BOS token then marsk the start of text tokens -->
* We use \n as SEP token, it does not appear in any of our prefixes -->
* We also tokenize SEP separately to avoid it being merged ( by the tokenizer) with either the end of the prefix or the begining of the suffix

In [ ]:
from typing import Dict, List, Optional, Union, Tuple, Iterable
from PIL import Image
import numpy as np
import torch
from torchvision import transforms


def add_image_tokens_to_prompt(prefix_prompt, bos_token, image_seq_len, image_token):
    # Quoting from the blog (https://huggingface.co/blog/paligemma#detailed-inference-process):
    #   The input text is tokenized normally.
    #   A <bos> token is added at the beginning, and an additional newline token (\n) is appended.
    #   This newline token is an essential part of the input prompt the model was trained with, so adding it explicitly ensures it's always there.
    #   The tokenized text is also prefixed with a fixed number of <image> tokens.
    # NOTE: from the paper it looks like the `\n` should be tokenized separately, but in the HF implementation this is not done.
    #       ref to HF implementation: https://github.com/huggingface/transformers/blob/7f79a97399bb52aad8460e1da2f36577d5dccfed/src/transformers/models/paligemma/processing_paligemma.py#L55-L73
    return f"{image_token * image_seq_len}{bos_token}{prefix_prompt}\n"

class PaliGemmaProcessor:
    "givne text and an image, create text token with placeholder for the image. "
    IMAGE_TOKEN = "<image>"

    def __init__(self,tokenizer, num_image_tokens: int, image_size: int):
        super().__init__()

        self.image_seq_length = num_image_tokens
        self.image_size = image_size
        self.image_transform =  transforms.Compose([                                                                                  
          transforms.Resize((self.image_size, self.image_size),                                                                    
                            interpolation=transforms.InterpolationMode.BICUBIC),                                                   
          transforms.ToTensor(),  # [0,255] PIL -> [0,1] [C,H,W] tensor                                                            
          transforms.Normalize(mean=IMAGENET_STANDARD_MEAN, std=IMAGENET_STANDARD_STD),                                            
      ])                                                                                                                           
                  

        #Tokeniser described here: https://github.com/google-research/big_vision/blob/main/big_vision/configs/proj/paligemma/README.md#tokenizer
        #We further extend its vocabulary with 1024 entries that represent coordinates in normalized image-space (<loc0000>...<loc1023>), and another with 128 entries (<seg000>...<seg127>) 
        # that are codewords used by a lightweight referring-expression segmentation vector-quantized variational auto-encoder (VQ-VAE) 
        #PaliGemma can do segmentation, detection
        # also, https://huggingface.co/blog/paligemma
        tokens_to_add = {'additional_special_tokens': [self.IMAGE_TOKEN]}
        tokenizer.add_special_tokens(tokens_to_add)
        EXTRA_TOKENS = [
            f"<loc{i:04d}>" for i in range(1024)
        ]  # These tokens are used for object detection (bounding boxes)
        EXTRA_TOKENS += [
            f"<seg{i:03d}>" for i in range(128)
        ]  # These tokens are used for object segmentation
        tokenizer.add_tokens(EXTRA_TOKENS)
        self.image_token_id = tokenizer.convert_tokens_to_ids(self.IMAGE_TOKEN)
        # We will add the BOS and EOS tokens ourselves
        tokenizer.add_bos_token = False
        tokenizer.add_eos_token = False

        self.tokenizer = tokenizer

    def __call__(
            self,
            text: List[str], 
            images: List[Image.Image],
            padding: str = "longest",
            truncation: bool = True,
    ) -> dict:
        assert len(images) == 1 and len(text) == 1, f"Received {len(images)} images for {len(text)} prompts."
        # rescale, normalize, convert to a tensor etc..
        pixel_values = torch.stack([self.image_transform(img) for img in images]) 

        # Prepend a `self.image_seq_length` number of image tokens to the prompt
        input_strings = [
            add_image_tokens_to_prompt(
                prefix_prompt=prompt,
                bos_token=self.tokenizer.bos_token,
                image_seq_len=self.image_seq_length,
                image_token=self.IMAGE_TOKEN,
            )
            for prompt in text
        ]
        # Returns the input_ids and attention_mask as PyTorch tensors
        inputs = self.tokenizer(
            input_strings,
            return_tensors="pt",
            padding=padding,
            truncation=truncation,
        )
        return_data = {"pixel_values": pixel_values, **inputs}
        return return_data